# Lifelines Modelling
This notebook builds and evaluates survival models to predict when customers are likely to place their next order. It starts with data loading and preprocessing, fits a Cox proportional hazards model, checks model assumptions, and then compares alternative accelerated failure time (AFT) models.

In [ ]:
import pandas as pd
from lifelines import CoxPHFitter, LogNormalAFTFitter, WeibullAFTFitter

from sme_kt_zh_collaboration_forecasting import modelling as sm
from sme_kt_zh_collaboration_forecasting import utils as su

## Prepare the Input Data
The raw sales data is loaded and deduplicated here. This creates a clean event history that can safely be used for train-test splitting and survival-data construction.

In [ ]:
df = su.read_sales_data("../data/sales_df.csv")
len_b = len(df)
df = df.drop_duplicates()
len_a = len(df)

print(f"Dropped {len_b - len_a} duplicates.")

## Build, Fit, and Rank with the Cox Model
This block engineers the prior-order feature, creates the shared train-test split used throughout the notebook, prepares the censored survival dataset, and fits the Cox proportional hazards model with the same covariates later used in the AFT models. The ranking outputs here are stored in reusable variables so the final comparison against Weibull and log-normal AFT is apples to apples.

In [ ]:
# 0. Ensure consistent ordering and num_prior_orders across the whole history
df = df.copy()
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(["customer", "date"])
df["num_prior_orders"] = df.groupby("customer").cumcount()

# 1. Split Raw Data
CUTOFF = "2024-12-31"
TOP_K = 50
df_train_raw, df_test_raw = sm.create_test_train(df, CUTOFF)

# 2. Initial Filtering
# We filter for >3 orders IN TRAIN to ensure we have enough history to model
df_train_raw = sm.filter_for_n_orders(df_train_raw, n=3)
train_customers = df_train_raw["customer"].unique()
df_test_raw = df_test_raw[df_test_raw["customer"].isin(train_customers)].copy()

# 3. Prepare Training Data for Cox
# This applies the censoring logic so the model knows who is currently 'silent'
train_final = sm.prepare_data(df_train_raw, CUTOFF)

# 4. Fit the Model
# We use num_prior_orders as a proxy for 'loyalty' / frequency
cph = CoxPHFitter(penalizer=0.1)

cph.fit(
    train_final,
    duration_col="duration",
    event_col="event",
    cluster_col="customer",
    strata=["customer_cat"],
    formula="num_prior_orders",
)

cph.print_summary()

# 5. Prepare Test Data
test_final = sm.prepare_data(df_test_raw, CUTOFF)

# 6. Priority list evaluation on test
prio = sm.predicted_vs_real_priorities(cph, test_final, customer_col="customer")
cox_ranking_summary = sm.summarize_top_k_predictions(prio, top_k=TOP_K)

print(
    f"Within the top {TOP_K}, {cox_ranking_summary['correct_predictions']} customers are in both lists."
)
print(f"Recall@{TOP_K}:    {cox_ranking_summary['recall_at_k']:.2%}")

## Check Cox Assumptions
After fitting the Cox model, this section runs diagnostic checks for the proportional hazards assumptions. The output helps determine whether the chosen features and stratification are compatible with the model assumptions.

In [ ]:
cph.check_assumptions(train_final, p_value_threshold=0.05, show_plots=True)

## Evaluate Cox Performance
This section computes concordance metrics for the Cox model on the same test cohort used by both AFT models. Together with the stored top-k ranking metrics, these values feed directly into the final comparison table.

In [ ]:
# Concordance evaluation on the shared test set. We calculate two test scores because we want to make sure it is correct.
cox_c_index_score = sm.c_index_on_test_via_score(cph, test_final)
cox_c_index_manual = sm.c_index_on_test_manual(cph, test_final)
cox_comparison_row = sm.comparison_row(
    model_name="Cox PH",
    c_index=cox_c_index_score,
    ranking_summary=cox_ranking_summary,
)

print("Test C-index (CoxPH, score): ", cox_c_index_score)
print("Test C-index (CoxPH, manual):", cox_c_index_manual)

## Compare Weibull and Log-Normal AFT Variants
This section reuses the same training cohort, test cohort, covariates, penalization level, and evaluation metrics used for the Cox model. That keeps the Weibull, log-normal, and Cox results directly comparable across concordance and top-k ranking quality.

In [ ]:
# ======================== WEIBULL AFT ========================
weibull_aft = WeibullAFTFitter(penalizer=0.1)
weibull_aft.fit(
    train_final,
    duration_col="duration",
    event_col="event",
    formula="num_prior_orders + customer_cat",
)

weibull_aft.print_summary()

prio_weibull = sm.predicted_vs_real_priorities_aft(
    weibull_aft,
    test_final,
    customer_col="customer",
)
weibull_ranking_summary = sm.summarize_top_k_predictions(
    prio_weibull,
    top_k=TOP_K,
)
weibull_c_index = sm.c_index_on_test_via_score(weibull_aft, test_final)
weibull_comparison_row = sm.comparison_row(
    model_name="Weibull AFT",
    c_index=weibull_c_index,
    ranking_summary=weibull_ranking_summary,
)

print(
    f"[Weibull AFT] Within the top {TOP_K}, "
    f"{weibull_ranking_summary['correct_predictions']} customers are in both lists."
)
print(f"[Weibull AFT] Recall@{TOP_K}:    {weibull_ranking_summary['recall_at_k']:.2%}")
print("Test C-index (Weibull AFT):", weibull_c_index)

# ======================== LOG-NORMAL AFT ========================
lognorm_aft = LogNormalAFTFitter(penalizer=0.1)
lognorm_aft.fit(
    train_final,
    duration_col="duration",
    event_col="event",
    formula="num_prior_orders + customer_cat",
)

lognorm_aft.print_summary()

prio_lognorm = sm.predicted_vs_real_priorities_aft(
    lognorm_aft,
    test_final,
    customer_col="customer",
)
lognorm_ranking_summary = sm.summarize_top_k_predictions(
    prio_lognorm,
    top_k=TOP_K,
)
lognorm_c_index = sm.c_index_on_test_via_score(lognorm_aft, test_final)
lognorm_comparison_row = sm.comparison_row(
    model_name="Log-Normal AFT",
    c_index=lognorm_c_index,
    ranking_summary=lognorm_ranking_summary,
)

print(
    f"[Log-Normal AFT] Within the top {TOP_K}, "
    f"{lognorm_ranking_summary['correct_predictions']} customers are in both lists."
)
print(
    f"[Log-Normal AFT] Recall@{TOP_K}:    {lognorm_ranking_summary['recall_at_k']:.2%}"
)
print("Test C-index (Log-Normal AFT):", lognorm_c_index)

### Evaluation Strategy: Focus on Recall @ k

We prioritize **Recall** over ranking metrics like the C-index. Our primary objective is to identify the specific cohort of customers likely to order within a defined time window, rather than achieving a perfect ordinal ranking of the entire database. Here, Recall represents our "hit rate": the proportion of customers in our **Top $k$ priority list** who actually transacted during the observation period.

**Performance Note:**
The current models achieve a Recall of approx. **62%**. 

* **As a Workshop Result:** This is a highly encouraging first milestone. It confirms that the available features contain a significant predictive signal and provides a clear uplift over a random outreach strategy.
* **Path to Production:** While 62% validates the approach, a production-grade system would likely require higher precision to optimize sales resources. Future iterations should focus on richer feature engineering (e.g., seasonality, product-category trends) to push this metric toward a more robust threshold for automated business decisions.

In [ ]:
comparison_df = pd.DataFrame(
    [
        cox_comparison_row,
        weibull_comparison_row,
        lognorm_comparison_row,
    ]
)
comparison_df[["c_index", "recall_at_k"]] = comparison_df[
    [
        "c_index",
        "recall_at_k",
    ]
].round(4)

comparison_df